# 🎧 NexTune — Bluetooth Headphones Price Prediction
## Exploratory Data Analysis (EDA)

**Dataset:** 219 products scraped from Amazon India | 40 features
**Goal:** Understand price drivers for wireless Bluetooth headphones in the Indian market

> Open in Colab → Runtime → Run all

In [ ]:
!pip install -q pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
print("✅ Ready")

In [ ]:
import os
if os.path.exists('/content'):
    URL = 'https://raw.githubusercontent.com/ESpoorthy/NexTune/main/data/final-merged-dataset.csv'
else:
    URL = '../data/final-merged-dataset.csv'

df = pd.read_csv(URL)

# Ensure binary columns are numeric
binary_cols = ['has_noise_cancellation','has_enc','has_gaming_mode','has_hi_res_audio',
               'has_spatial_audio','has_low_latency','has_fast_charge','has_touch_control',
               'has_dual_pairing','has_usb_c','has_premium_codec','has_voice_assistant']
for c in binary_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0).astype(int)

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head(3)

## 1. Dataset Overview

In [ ]:
print("=" * 55)
print("DATASET OVERVIEW")
print("=" * 55)
print(f"  Total products    : {len(df)}")
print(f"  Total features    : {len(df.columns)}")
print(f"  Unique brands     : {df['brand'].nunique()}")
print(f"  Price range       : ₹{df['price_inr'].min():.0f} – ₹{df['price_inr'].max():.0f}")
print(f"  Median price      : ₹{df['price_inr'].median():.0f}")
print(f"  Mean price        : ₹{df['price_inr'].mean():.0f}")
print(f"  Avg rating        : {df['rating'].mean():.2f} / 5.0")
print("=" * 55)
print("\nMissing values (top 10):")
missing = df.isnull().sum().sort_values(ascending=False)
print(missing[missing > 0].head(10).to_string())

## 2. Price Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram
axes[0].hist(df['price_inr'].dropna(), bins=40, color='#378ADD', edgecolor='white')
axes[0].axvline(df['price_inr'].median(), color='red', linestyle='--', linewidth=2,
                label=f"Median ₹{df['price_inr'].median():.0f}")
axes[0].axvline(df['price_inr'].mean(), color='orange', linestyle='--', linewidth=2,
                label=f"Mean ₹{df['price_inr'].mean():.0f}")
axes[0].set_title('Price Distribution', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Price (INR)')
axes[0].legend()

# Box plot
axes[1].boxplot(df['price_inr'].dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor='#378ADD', alpha=0.6))
axes[1].set_title('Price Box Plot', fontweight='bold', fontsize=13)
axes[1].set_ylabel('Price (INR)')

# Log scale histogram
axes[2].hist(np.log1p(df['price_inr'].dropna()), bins=40, color='#1D9E75', edgecolor='white')
axes[2].set_title('Price Distribution (Log Scale)', fontweight='bold', fontsize=13)
axes[2].set_xlabel('log(Price + 1)')

plt.suptitle('Price Analysis — 219 Products', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(df['price_inr'].describe().round(0).to_string())

## 3. Category & Brand Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Category count
cat = df['category'].value_counts()
colors = ['#378ADD','#1D9E75','#D85A30','#BA7517']
axes[0].bar(cat.index, cat.values, color=colors[:len(cat)])
axes[0].set_title('Products by Category', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=20)
for i, v in enumerate(cat.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontweight='bold')

# Price by category
df.boxplot(column='price_inr', by='category', ax=axes[1],
           boxprops=dict(color='#378ADD'), medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Price by Category', fontweight='bold', fontsize=13)
axes[1].set_xlabel('')
plt.sca(axes[1]); plt.xticks(rotation=20)
plt.suptitle('')

# Top 12 brands
top_brands = df['brand'].value_counts().head(12)
axes[2].barh(top_brands.index[::-1], top_brands.values[::-1], color='#378ADD')
axes[2].set_title('Top 12 Brands by Product Count', fontweight='bold', fontsize=13)
axes[2].set_xlabel('Number of Products')

plt.tight_layout()
plt.show()

print("\nCategory stats:")
print(df.groupby('category')['price_inr'].agg(['count','mean','median','min','max']).round(0).to_string())

## 4. Country of Origin Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Country pie chart
country = df['country_of_origin'].fillna('Unknown').value_counts()
colors_pie = ['#378ADD','#D85A30','#1D9E75','#BA7517','#D4537E','#888888']
axes[0].pie(country.values, labels=country.index, autopct='%1.1f%%',
            colors=colors_pie[:len(country)], startangle=90)
axes[0].set_title('Products by Country of Origin', fontweight='bold', fontsize=13)

# Price by country
country_price = df.groupby('country_of_origin')['price_inr'].median().sort_values(ascending=False)
axes[1].bar(country_price.index, country_price.values, color='#1D9E75')
axes[1].set_title('Median Price by Country of Origin', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Country')
axes[1].set_ylabel('Median Price (INR)')
axes[1].tick_params(axis='x', rotation=20)
for i, v in enumerate(country_price.values):
    axes[1].text(i, v + 20, f'₹{v:.0f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nCountry breakdown:")
print(df.groupby('country_of_origin')['price_inr'].agg(['count','mean','median']).round(0).to_string())

## 5. Feature Analysis — What Features Drive Price?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Feature prevalence
features = {
    'ANC': 'has_noise_cancellation',
    'ENC': 'has_enc',
    'Fast Charge': 'has_fast_charge',
    'Gaming Mode': 'has_gaming_mode',
    'Touch Control': 'has_touch_control',
    'Low Latency': 'has_low_latency',
    'Dual Pairing': 'has_dual_pairing',
    'Spatial Audio': 'has_spatial_audio',
    'Voice Assist': 'has_voice_assistant',
    'USB-C': 'has_usb_c',
    'Hi-Res Audio': 'has_hi_res_audio',
}
feat_counts = {name: int(df[col].fillna(0).sum()) for name, col in features.items() if col in df.columns}
feat_df = pd.Series(feat_counts).sort_values(ascending=True)

axes[0].barh(feat_df.index, feat_df.values, color='#378ADD')
axes[0].set_title('Feature Prevalence (# Products)', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Number of Products')
for i, v in enumerate(feat_df.values):
    axes[0].text(v + 0.3, i, str(v), va='center', fontweight='bold')

# ANC vs non-ANC price
anc_yes = df[df['has_noise_cancellation'] == 1]['price_inr'].dropna()
anc_no  = df[df['has_noise_cancellation'] == 0]['price_inr'].dropna()
axes[1].boxplot([anc_no, anc_yes], labels=['No ANC', 'Has ANC'],
                patch_artist=True,
                boxprops=dict(facecolor='#378ADD', alpha=0.6),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Price: ANC vs Non-ANC', fontweight='bold', fontsize=13)
axes[1].set_ylabel('Price (INR)')
axes[1].text(1, anc_no.median() + 50, f'₹{anc_no.median():.0f}', ha='center', color='red', fontweight='bold')
axes[1].text(2, anc_yes.median() + 50, f'₹{anc_yes.median():.0f}', ha='center', color='red', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nMedian price WITH ANC    : ₹{anc_yes.median():.0f}")
print(f"Median price WITHOUT ANC : ₹{anc_no.median():.0f}")
print(f"Price premium for ANC    : ₹{anc_yes.median() - anc_no.median():.0f}")

## 6. Battery Life & Bluetooth Version Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Battery life distribution
batt = df['battery_life_hrs'].dropna()
axes[0].hist(batt, bins=30, color='#1D9E75', edgecolor='white')
axes[0].axvline(batt.median(), color='red', linestyle='--', linewidth=2,
                label=f'Median: {batt.median():.0f}h')
axes[0].set_title('Battery Life Distribution', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Battery Life (hours)')
axes[0].legend()

# Battery vs Price scatter
axes[1].scatter(df['battery_life_hrs'], df['price_inr'], alpha=0.5, color='#378ADD', s=50)
axes[1].set_title('Battery Life vs Price', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Battery Life (hours)')
axes[1].set_ylabel('Price (INR)')

# Bluetooth version distribution
bt = df['bluetooth_version'].dropna()
bt_counts = bt.value_counts().sort_index()
axes[2].bar([str(x) for x in bt_counts.index], bt_counts.values, color='#D85A30')
axes[2].set_title('Bluetooth Version Distribution', fontweight='bold', fontsize=13)
axes[2].set_xlabel('Bluetooth Version')
axes[2].set_ylabel('Count')
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

print(f"Battery life — mean: {batt.mean():.1f}h | median: {batt.median():.1f}h | max: {batt.max():.0f}h")
print(f"\nBluetooth version breakdown:")
print(df['bluetooth_version'].value_counts().sort_index().to_string())

## 7. Rating Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Rating distribution
rat = df['rating'].dropna()
axes[0].hist(rat, bins=20, color='#BA7517', edgecolor='white')
axes[0].axvline(rat.mean(), color='red', linestyle='--', linewidth=2,
                label=f'Mean: {rat.mean():.2f}')
axes[0].set_title('Rating Distribution', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Rating (out of 5)')
axes[0].legend()

# Rating vs Price
axes[1].scatter(df['rating'], df['price_inr'], alpha=0.5, color='#BA7517', s=50)
axes[1].set_title('Rating vs Price', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Rating')
axes[1].set_ylabel('Price (INR)')

plt.tight_layout()
plt.show()

print(f"Rating — mean: {rat.mean():.2f} | median: {rat.median():.2f}")
print(f"Products rated ≥ 4.0: {(rat >= 4.0).sum()} ({(rat >= 4.0).mean()*100:.1f}%)")

## 8. Correlation Heatmap

In [ ]:
num_cols = ['price_inr','rating','review_count','battery_life_hrs','driver_size_mm',
            'bluetooth_version','has_noise_cancellation','has_enc','has_fast_charge',
            'has_gaming_mode','has_touch_control','has_low_latency']
corr_df = df[[c for c in num_cols if c in df.columns]].apply(pd.to_numeric, errors='coerce').corr()

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_df, dtype=bool))
sns.heatmap(corr_df, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, mask=mask, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nCorrelation with price_inr (sorted):")
print(corr_df['price_inr'].sort_values(ascending=False).drop('price_inr').round(3).to_string())

## 9. Key EDA Insights Summary

In [ ]:
print("=" * 60)
print("KEY EDA INSIGHTS — NexTune Headphones Dataset")
print("=" * 60)
print(f"\n📦 Dataset")
print(f"   {len(df)} products | {df['brand'].nunique()} brands | 40 features")
print(f"\n💰 Price")
print(f"   Range  : ₹{df['price_inr'].min():.0f} – ₹{df['price_inr'].max():.0f}")
print(f"   Median : ₹{df['price_inr'].median():.0f}  |  Mean: ₹{df['price_inr'].mean():.0f}")
print(f"   75% of products priced below ₹{df['price_inr'].quantile(0.75):.0f}")
print(f"\n🏷️  Categories")
for cat, cnt in df['category'].value_counts().items():
    print(f"   {cat}: {cnt} ({cnt/len(df)*100:.1f}%)")
print(f"\n🌍 Country of Origin")
for c, cnt in df['country_of_origin'].value_counts().items():
    print(f"   {c}: {cnt}")
print(f"\n🔋 Battery Life")
batt = df['battery_life_hrs'].dropna()
print(f"   Mean: {batt.mean():.1f}h | Median: {batt.median():.1f}h | Max: {batt.max():.0f}h")
print(f"\n⭐ Ratings")
rat = df['rating'].dropna()
print(f"   Mean: {rat.mean():.2f} | {(rat>=4.0).sum()} products rated ≥ 4.0")
print(f"\n🎯 Key Features")
anc_med = df[df['has_noise_cancellation']==1]['price_inr'].median()
no_anc_med = df[df['has_noise_cancellation']==0]['price_inr'].median()
print(f"   ANC products: {int(df['has_noise_cancellation'].fillna(0).sum())} — median price ₹{anc_med:.0f} vs ₹{no_anc_med:.0f} without")
print(f"   ENC: {int(df['has_enc'].sum())} | Fast Charge: {int(df['has_fast_charge'].fillna(0).sum())} | Gaming Mode: {int(df['has_gaming_mode'].sum())}")
print("=" * 60)